# Notebook 04: Modeling - Prédiction des Complications Post-Vaccination/Post-Traitement

## Objectif
Créer et entraîner plusieurs modèles de Machine Learning pour prédire les effets indésirables graves :
- **ML Classique**: XGBoost et Random Forest
- **Graph ML**: GNN (Graph Neural Networks)
- **NLP**: BERT pour analyse des symptômes textuels
- **Fusion**: Combinaison des modèles pour prédiction finale

## Données
Utilisation du dataset `dataset_for_modeling.csv` préparé dans le Notebook 03.

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
from transformers import BertTokenizer, BertModel
import shap
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("✅ Toutes les bibliothèques importées avec succès!")

In [ ]:
# Chargement des données
DATA_PATH = "../data/final/dataset_for_modeling.csv"
df = pd.read_csv(DATA_PATH)

print(f"✅ Dataset chargé: {df.shape[0]} lignes, {df.shape[1]} colonnes")
print("\n📊 Aperçu des données:")
print(df.head())

print("\n📈 Statistiques descriptives:")
print(df.describe())

# Vérification des valeurs manquantes
print("\n❓ Valeurs manquantes par colonne:")
print(df.isnull().sum())

In [ ]:
# Préparation des données pour les différents modèles

# Target variable (on utilise target_binary pour classification binaire)
target = 'target_binary'

# Features pour ML classique (numériques uniquement)
numerical_features = [
    'age', 'weight_kg', 'bmi', 'n_drugs', 'n_symptoms',
    'n_allergies', 'n_chronic_diseases', 'score_risque_interaction',
    'vulnerability_score', 'severity_score'
]

# Features textuelles pour BERT
text_features = ['symptoms_text']

# Vérification des colonnes disponibles
available_features = [col for col in numerical_features if col in df.columns]
print(f"✅ Features numériques disponibles: {available_features}")

# Séparation features et target
X_numerical = df[available_features]
y = df[target]

print(f"✅ Shape X_numerical: {X_numerical.shape}, y: {y.shape}")

# Gestion des valeurs manquantes
X_numerical = X_numerical.fillna(X_numerical.mean())
print("✅ Valeurs manquantes traitées")

# Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_numerical)
X_scaled = pd.DataFrame(X_scaled, columns=available_features)

print("✅ Données normalisées")

In [ ]:
# Split des données
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"✅ Split terminé:")
print(f"  - Train: {X_train.shape[0]} échantillons")
print(f"  - Test: {X_test.shape[0]} échantillons")
print(f"  - Distribution classes train: {y_train.value_counts().to_dict()}")
print(f"  - Distribution classes test: {y_test.value_counts().to_dict()}")

## 1. Modèles Machine Learning Classiques

In [ ]:
# Modèle XGBoost
print("🚀 Entraînement du modèle XGBoost...")

xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

# Entraînement avec validation croisée
xgb_cv_scores = cross_val_score(xgb_model, X_train, y_train, cv=5, scoring='roc_auc')
print(f"✅ XGBoost - CV AUC: {xgb_cv_scores.mean():.3f} (+/- {xgb_cv_scores.std() * 2:.3f})")

# Entraînement final
xgb_model.fit(X_train, y_train)

# Prédictions
xgb_pred = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

# Évaluation
print("\n📊 Résultats XGBoost sur test:")
print(classification_report(y_test, xgb_pred))
print(f"AUC-ROC: {roc_auc_score(y_test, xgb_proba):.3f}")

# Matrice de confusion
plt.figure(figsize=(6, 4))
cm = confusion_matrix(y_test, xgb_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Matrice de Confusion - XGBoost')
plt.ylabel('Vrai label')
plt.xlabel('Prédiction')
plt.show()

In [ ]:
# Modèle Random Forest
print("\n🚀 Entraînement du modèle Random Forest...")

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight='balanced'
)

# Entraînement avec validation croisée
rf_cv_scores = cross_val_score(rf_model, X_train, y_train, cv=5, scoring='roc_auc')
print(f"✅ Random Forest - CV AUC: {rf_cv_scores.mean():.3f} (+/- {rf_cv_scores.std() * 2:.3f})")

# Entraînement final
rf_model.fit(X_train, y_train)

# Prédictions
rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

# Évaluation
print("\n📊 Résultats Random Forest sur test:")
print(classification_report(y_test, rf_pred))
print(f"AUC-ROC: {roc_auc_score(y_test, rf_proba):.3f}")

# Matrice de confusion
plt.figure(figsize=(6, 4))
cm = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens')
plt.title('Matrice de Confusion - Random Forest')
plt.ylabel('Vrai label')
plt.xlabel('Prédiction')
plt.show()

# Importance des features
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=feature_importance.head(10))
plt.title('Top 10 Features Importantes - Random Forest')
plt.show()

## 2. Modèle Graph Neural Network (GNN)

Pour le Graph ML, nous allons créer un graphe où :
- **Nœuds patients**: caractéristiques démographiques et médicales
- **Nœuds traitements**: informations sur les vaccins/médicaments
- **Arêtes**: relations patient-traitement avec historique des effets

In [ ]:
# Création des données pour le GNN
print("🔗 Construction du graphe pour GNN...")

# Vérification des colonnes nécessaires
if 'patient_id' not in df.columns or 'treatment_id' not in df.columns:
    print("❌ Colonnes patient_id ou treatment_id manquantes. Création...")
    df['patient_id'] = range(len(df))
    df['treatment_id'] = pd.factorize(df.get('vaccine_or_drug', 'unknown'))[0]

# Nombre de nœuds
n_patients = df['patient_id'].nunique()
n_treatments = df['treatment_id'].nunique()
total_nodes = n_patients + n_treatments

print(f"📊 Graphe: {n_patients} patients + {n_treatments} traitements = {total_nodes} nœuds")

# Création des features des nœuds
patient_features = []
treatment_features = []

# Features pour les patients (mêmes que ML classique)
patient_cols = available_features
for pid in range(n_patients):
    patient_data = df[df['patient_id'] == pid][patient_cols].iloc[0] if len(df[df['patient_id'] == pid]) > 0 else df[patient_cols].mean()
    patient_features.append(patient_data.values)

# Features pour les traitements (simplifiées)
treatment_cols = ['n_drugs', 'severity_score'] if 'n_drugs' in df.columns else ['severity_score']
for tid in range(n_treatments):
    treatment_data = df[df['treatment_id'] == tid][treatment_cols].mean() if len(df[df['treatment_id'] == tid]) > 0 else df[treatment_cols].mean()
    treatment_features.append(treatment_data.values)

# Concaténation des features
node_features = np.vstack([np.array(patient_features), np.array(treatment_features)])
node_features = torch.tensor(node_features, dtype=torch.float)

print(f"✅ Features des nœuds créées: {node_features.shape}")

# Création des arêtes (edges)
edges = []
for _, row in df.iterrows():
    patient_idx = row['patient_id']
    treatment_idx = n_patients + row['treatment_id']  # offset pour les traitements
    edges.append([patient_idx, treatment_idx])
    edges.append([treatment_idx, patient_idx])  # graphe non dirigé

edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

print(f"✅ Arêtes créées: {edge_index.shape[1]} connexions")

# Labels pour les patients uniquement
patient_labels = []
for pid in range(n_patients):
    patient_rows = df[df['patient_id'] == pid]
    if len(patient_rows) > 0:
        label = patient_rows[target].iloc[0]
    else:
        label = 0  # défaut
    patient_labels.append(label)

y_graph = torch.tensor(patient_labels, dtype=torch.long)

# Création de l'objet Data pour PyTorch Geometric
graph_data = Data(x=node_features, edge_index=edge_index, y=y_graph)

print(f"✅ Objet graphe créé: {graph_data}")
print(f"  - Nœuds: {graph_data.num_nodes}")
print(f"  - Arêtes: {graph_data.num_edges}")
print(f"  - Features par nœud: {graph_data.num_node_features}")

In [ ]:
# Définition du modèle GNN
class GNNModel(nn.Module):
    def __init__(self, num_node_features, hidden_channels, num_classes):
        super(GNNModel, self).__init__()
        self.conv1 = GCNConv(num_node_features, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.conv3 = GCNConv(hidden_channels, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x, edge_index, batch=None):
        x = self.conv1(x, edge_index)
        x = x.relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        x = x.relu()
        x = self.dropout(x)
        x = self.conv3(x, edge_index)
        return x

# Initialisation du modèle
gnn_model = GNNModel(
    num_node_features=node_features.shape[1],
    hidden_channels=64,
    num_classes=2
)

print(f"✅ Modèle GNN créé: {gnn_model}")

# Optimiseur et fonction de perte
optimizer = optim.Adam(gnn_model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

# Split pour le graphe (utilisation des indices des patients)
train_indices, test_indices = train_test_split(
    range(n_patients), test_size=0.2, random_state=42, stratify=y_graph
)

print(f"✅ Split GNN: {len(train_indices)} train, {len(test_indices)} test")

In [ ]:
# Entraînement du GNN
print("🚀 Entraînement du modèle GNN...")

def train_gnn(model, data, train_indices, optimizer, criterion, epochs=50):
    model.train()
    losses = []

    for epoch in range(epochs):
        optimizer.zero_grad()

        # Forward pass
        out = model(data.x, data.edge_index)

        # Prédictions pour les patients uniquement
        patient_out = out[:n_patients]
        train_out = patient_out[train_indices]

        # Labels d'entraînement
        train_labels = data.y[train_indices]

        # Calcul de la perte
        loss = criterion(train_out, train_labels)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1:3d}/{epochs}, Loss: {loss.item():.4f}")

    return losses

# Entraînement
gnn_losses = train_gnn(gnn_model, graph_data, train_indices, optimizer, criterion, epochs=50)

# Visualisation de la perte
plt.figure(figsize=(8, 4))
plt.plot(gnn_losses)
plt.title('Évolution de la perte - GNN')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

# Évaluation du GNN
gnn_model.eval()
with torch.no_grad():
    out = gnn_model(graph_data.x, graph_data.edge_index)
    patient_out = out[:n_patients]

    # Prédictions
    gnn_pred = patient_out[test_indices].argmax(dim=1).numpy()
    gnn_proba = torch.softmax(patient_out[test_indices], dim=1)[:, 1].numpy()

    # Labels réels
    y_test_gnn = graph_data.y[test_indices].numpy()

# Résultats
print("\n📊 Résultats GNN sur test:")
print(classification_report(y_test_gnn, gnn_pred))
print(f"AUC-ROC: {roc_auc_score(y_test_gnn, gnn_proba):.3f}")

# Matrice de confusion
plt.figure(figsize=(6, 4))
cm = confusion_matrix(y_test_gnn, gnn_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples')
plt.title('Matrice de Confusion - GNN')
plt.ylabel('Vrai label')
plt.xlabel('Prédiction')
plt.show()

## 3. Modèle BERT pour Analyse NLP des Symptômes

Utilisation de BERT pour extraire des embeddings des descriptions textuelles des symptômes.

In [ ]:
# Modèle BERT pour les symptômes textuels
print("🧠 Chargement de BERT pour analyse NLP...")

# Initialisation du tokenizer et modèle BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')

# Fonction pour extraire les embeddings BERT
def get_bert_embeddings(texts, max_length=128):
    embeddings = []

    for text in texts:
        # Prétraitement du texte
        text = str(text) if pd.notna(text) else ""
        if len(text) == 0:
            text = "no symptoms reported"

        # Tokenization
        inputs = tokenizer(text, return_tensors='pt', max_length=max_length,
                          padding='max_length', truncation=True)

        # Extraction des embeddings
        with torch.no_grad():
            outputs = bert_model(**inputs)
            # Utilisation de l'embedding [CLS] comme représentation du texte
            embedding = outputs.last_hidden_state[:, 0, :].squeeze().numpy()

        embeddings.append(embedding)

    return np.array(embeddings)

# Application sur les données d'entraînement et test
print("🔄 Extraction des embeddings BERT pour les données d'entraînement...")
X_text_train = get_bert_embeddings(df.iloc[X_train.index]['symptoms_text'].values)
print(f"✅ Embeddings train: {X_text_train.shape}")

print("🔄 Extraction des embeddings BERT pour les données de test...")
X_text_test = get_bert_embeddings(df.iloc[X_test.index]['symptoms_text'].values)
print(f"✅ Embeddings test: {X_text_test.shape}")

# Modèle de classification basé sur BERT
class BERTClassifier(nn.Module):
    def __init__(self, bert_model, hidden_size=768, num_classes=2):
        super(BERTClassifier, self).__init__()
        self.bert = bert_model
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

# Création du modèle BERT classifier
bert_classifier = BERTClassifier(bert_model)
optimizer_bert = optim.Adam(bert_classifier.parameters(), lr=2e-5)
criterion_bert = nn.CrossEntropyLoss()

print("✅ Modèle BERT classifier créé")

In [ ]:
# Préparation des données tokenisées pour BERT
def tokenize_texts(texts, max_length=128):
    tokenized = []
    for text in texts:
        text = str(text) if pd.notna(text) else "no symptoms reported"
        inputs = tokenizer(text, return_tensors='pt', max_length=max_length,
                          padding='max_length', truncation=True)
        tokenized.append({
            'input_ids': inputs['input_ids'].squeeze(),
            'attention_mask': inputs['attention_mask'].squeeze()
        })
    return tokenized

print("🔄 Tokenization des textes pour BERT...")
train_tokenized = tokenize_texts(df.iloc[X_train.index]['symptoms_text'].values)
test_tokenized = tokenize_texts(df.iloc[X_test.index]['symptoms_text'].values)

# Conversion en tensors
train_input_ids = torch.stack([item['input_ids'] for item in train_tokenized])
train_attention_mask = torch.stack([item['attention_mask'] for item in train_tokenized])
train_labels_bert = torch.tensor(y_train.values, dtype=torch.long)

test_input_ids = torch.stack([item['input_ids'] for item in test_tokenized])
test_attention_mask = torch.stack([item['attention_mask'] for item in test_tokenized])
test_labels_bert = torch.tensor(y_test.values, dtype=torch.long)

print(f"✅ Données BERT préparées: {len(train_tokenized)} train, {len(test_tokenized)} test")

# Entraînement du modèle BERT
print("🚀 Entraînement du modèle BERT...")

def train_bert(model, train_input_ids, train_attention_mask, train_labels,
               optimizer, criterion, epochs=3, batch_size=16):
    model.train()
    losses = []

    for epoch in range(epochs):
        epoch_loss = 0
        n_batches = len(train_input_ids) // batch_size

        for i in range(0, len(train_input_ids), batch_size):
            batch_input_ids = train_input_ids[i:i+batch_size]
            batch_attention_mask = train_attention_mask[i:i+batch_size]
            batch_labels = train_labels[i:i+batch_size]

            optimizer.zero_grad()

            outputs = model(batch_input_ids, batch_attention_mask)
            loss = criterion(outputs, batch_labels)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        avg_loss = epoch_loss / n_batches
        losses.append(avg_loss)
        print(f"Epoch {epoch+1:2d}/{epochs}, Loss: {avg_loss:.4f}")

    return losses

# Entraînement (epochs réduit pour éviter le surapprentissage)
bert_losses = train_bert(bert_classifier, train_input_ids, train_attention_mask,
                        train_labels_bert, optimizer_bert, criterion_bert, epochs=3)

# Visualisation de la perte
plt.figure(figsize=(8, 4))
plt.plot(bert_losses)
plt.title('Évolution de la perte - BERT')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

# Évaluation BERT
bert_classifier.eval()
with torch.no_grad():
    test_outputs = bert_classifier(test_input_ids, test_attention_mask)
    bert_pred = torch.argmax(test_outputs, dim=1).numpy()
    bert_proba = torch.softmax(test_outputs, dim=1)[:, 1].numpy()

print("\n📊 Résultats BERT sur test:")
print(classification_report(y_test, bert_pred))
print(f"AUC-ROC: {roc_auc_score(y_test, bert_proba):.3f}")

# Matrice de confusion
plt.figure(figsize=(6, 4))
cm = confusion_matrix(y_test, bert_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges')
plt.title('Matrice de Confusion - BERT')
plt.ylabel('Vrai label')
plt.xlabel('Prédiction')
plt.show()

## 4. Modèle de Fusion

Combinaison des prédictions des différents modèles pour une prédiction finale plus robuste.

In [ ]:
# Modèle de Fusion
print("🔀 Création du modèle de fusion...")

# Collecte des prédictions de probabilité de chaque modèle
fusion_features = pd.DataFrame({
    'xgb_proba': xgb_proba,
    'rf_proba': rf_proba,
    'bert_proba': bert_proba,
    'gnn_proba': gnn_proba
})

print(f"✅ Features de fusion: {fusion_features.shape}")
print(f"📊 Corrélations entre modèles:")
print(fusion_features.corr())

# Modèle de fusion simple: moyenne pondérée
weights = {
    'xgb': 0.3,
    'rf': 0.2,
    'bert': 0.25,
    'gnn': 0.25
}

fusion_proba = (
    weights['xgb'] * fusion_features['xgb_proba'] +
    weights['rf'] * fusion_features['rf_proba'] +
    weights['bert'] * fusion_features['bert_proba'] +
    weights['gnn'] * fusion_features['gnn_proba']
)

fusion_pred = (fusion_proba > 0.5).astype(int)

print("\n📊 Résultats du modèle de fusion:")
print(classification_report(y_test, fusion_pred))
print(f"AUC-ROC: {roc_auc_score(y_test, fusion_proba):.3f}")

# Matrice de confusion
plt.figure(figsize=(6, 4))
cm = confusion_matrix(y_test, fusion_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds')
plt.title('Matrice de Confusion - Modèle de Fusion')
plt.ylabel('Vrai label')
plt.xlabel('Prédiction')
plt.show()

# Comparaison des performances
models = ['XGBoost', 'Random Forest', 'BERT', 'GNN', 'Fusion']
auc_scores = [
    roc_auc_score(y_test, xgb_proba),
    roc_auc_score(y_test, rf_proba),
    roc_auc_score(y_test, bert_proba),
    roc_auc_score(y_test, gnn_proba),
    roc_auc_score(y_test, fusion_proba)
]

plt.figure(figsize=(10, 6))
bars = plt.bar(models, auc_scores, color=['blue', 'green', 'orange', 'purple', 'red'])
plt.title('Comparaison des AUC-ROC des modèles')
plt.ylabel('AUC-ROC')
plt.ylim(0.5, 1.0)

for bar, score in zip(bars, auc_scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{score:.3f}', ha='center', va='bottom')

plt.show()

# Sauvegarde des modèles
import joblib

models_dict = {
    'xgb_model': xgb_model,
    'rf_model': rf_model,
    'gnn_model': gnn_model,
    'bert_classifier': bert_classifier,
    'scaler': scaler,
    'fusion_weights': weights
}

joblib.dump(models_dict, '../models/trained_models.pkl')
print("💾 Modèles sauvegardés dans '../models/trained_models.pkl'")

# Création du dossier models s'il n'existe pas
os.makedirs('../models', exist_ok=True)
joblib.dump(models_dict, '../models/trained_models.pkl')
print("✅ Sauvegarde terminée!")